[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/orientation/mirror_energy/mirror_energy.ipynb)

# Energy in a Mirror

This example computes the rotational kinetic energy of a rigid distribution of masses turning on a sphere and of its mirror image. Reflecting the body and its motion in a mirror reverses the sense of rotation, yet the mirrored body carries the same positive kinetic energy whether the reflection is applied to the inertia map or the inertia is built anew from the reflected masses.

On the unit sphere in three-dimensional geometric algebra, mass positions are modelled as antivectors with ***intrinsic*** orientation and turning rates as bivectors with ***extrinsic*** orientation. Each mass moves along `point.commutator(rate)`, and joining its weighted position with that velocity gives its intrinsic angular momentum. The term ***duality map*** here means a linear map between opposite orientations, such as the body's inertia mapping extrinsic rates to intrinsic momenta, so half the pairing `rate & momentum` yields an extrinsic scalar energy. Because `mirror >> x` transforms each element according to its orientation, reflecting the inertia map on input and output agrees with rebuilding it from the reflected masses, whereas the orientation-blind sandwich `mirror * x.involute() * mirror.inverse()` flips the momentum's sign and produces a negative energy.

In Gibbs–Heaviside vector notation, the inertia tensor $\mathsf I = \sum_i m_i\,(|\mathbf r_i|^2\,\mathbf I - \mathbf r_i\mathbf r_i^{\mathsf T})$ maps the angular velocity $\boldsymbol\omega$ to the angular momentum $\mathbf L = \mathsf I\boldsymbol\omega$ with kinetic energy $T = \tfrac12\,\boldsymbol\omega \cdot \mathbf L$; under an orthogonal reflection $M$, the polar positions transform as $M\mathbf r_i$ and $\mathsf I$ transforms as $M\mathsf I M^{\mathsf T}$, whereas the axial vectors $\boldsymbol\omega$ and $\mathbf L$ require explicit determinant factors $(\det M)\,M\boldsymbol\omega$ and $(\det M)\,M\mathbf L$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, stack
from numga.algebras import Spherical3D
from examples.util.animation import show_animation
from examples.orientation.mirror_energy import render

np.set_printoptions(precision=3, suppress=True)

ga = Spherical3D
mv = NumpyContext(ga).multivector
# Trailing type comments mark + extrinsic and - intrinsic orientation.
Rate = ga.gatype.bivector()                   # Bivector+

## Kinetic energy of the turning masses

Three unequal masses sit at fixed positions on the unit sphere and turn together about a shared plane of rotation. Summing each mass's contribution `(masses * points).regressive(points.commutator(Rate))` gives the body's inertia map from turning rates to angular momenta, and half the pairing `rate & momentum` evaluates the kinetic energy.

In [ ]:
points = mv.vector([[1, 0, 0], [0, 0.8, 0.6], [-0.6, 0, -0.8]]).dual()   # [3] Point-
masses = np.array([1, 1.5, 0.5])
rate = mv.xy                                                              # [] Rate+

In [ ]:
# Inertia maps the body's turning rate to its intrinsic angular momentum.
inertia = (masses * points).regressive(points.commutator(Rate)).sum(axis=0)  # [] Momentum- <- Rate+
momentum = inertia(rate)                                         # [] Momentum-
energy = (rate & momentum) / 2                                   # [] Scalar+

print("Energy")
print(energy.coefficient_table())

## The mirrored body

Conjugating the inertia map on its input and output with the mirror plane, `mirror >> inertia(mirror << Rate)`, transforms the map into the reflected frame and matches the inertia assembled directly from the reflected masses `mirror >> points`, so both routes give the mirrored body the same positive kinetic energy as the original.

In [ ]:
mirror = mv.x                                                    # [] Plane+
n_frames = 81
frame_duration_ms = 40
times = np.linspace(0, 2 * np.pi, n_frames, endpoint=False)

In [ ]:
# Reflect the turning rate, and conjugate the inertia map on both input and output.
reflected_rate = mirror >> rate                                  # [] Rate+
reflected_inertia = mirror >> inertia(mirror << Rate)            # [] Momentum- <- Rate+
reflected_energy = (reflected_rate & reflected_inertia(reflected_rate)) / 2  # [] Scalar+

# Rebuilding the inertia from the reflected masses produces the same map and energy.
reflected_points = mirror >> points                              # [3] Point-
rebuilt_inertia = (masses * reflected_points).regressive(reflected_points.commutator(Rate)).sum(axis=0)  # [] Momentum- <- Rate+
rebuilt_energy = (reflected_rate & rebuilt_inertia(reflected_rate)) / 2  # [] Scalar+

# Evolve the spherical motion over one full turn alongside its mirror image.
rotors = (rate * (-times[:, None] / 2)).exp()                    # [n_frames, 1] Rotor+
positions = rotors >> points                                     # [n_frames, 3] Point-
velocities = positions.commutator(rate)                          # [n_frames, 3] Point-
mirrored_positions = stack([positions, mirror >> positions], axis=1)    # [n_frames, 2, 3] Point-
mirrored_velocities = stack([velocities, mirror >> velocities], axis=1)  # [n_frames, 2, 3] Point-

print("Reflected energy")
print(reflected_energy.coefficient_table())
print("Rebuilt energy")
print(rebuilt_energy.coefficient_table())
show_animation(render.motion_animation(mirrored_positions, mirrored_velocities, masses), "mirror_energy", frame_duration_ms);

In the animation, the right-hand sphere shows the reflection of the left-hand body across the vertical mirror plane and turns in the opposite sense. Matching colours identify each mass and its mirror partner along their closed orbital paths, ball areas scale with mass, grey rods connect the three masses to show the reflected handedness, and coloured arrows show the tangent velocities.

## The sandwich without orientation

Applying the grade-involuted sandwich `mirror * x.involute() * mirror.inverse()` treats every multivector as extrinsic regardless of how it was constructed. That uniform formula reflects the extrinsic turning rate to `mirror >> rate` while flipping the sign of the intrinsic angular momentum relative to `mirror >> momentum`, so pairing the two gives the mirrored body a negative kinetic energy.

In [ ]:
# Uniform grade-involuted conjugation flips the sign of intrinsic elements under reflection.
sandwich_momentum = mirror * momentum.involute() * mirror.inverse()  # [] Momentum-
sandwich_rate = mirror * rate.involute() * mirror.inverse()          # [] Rate+
sandwich_energy = (sandwich_rate & sandwich_momentum) / 2            # [] Scalar+

print("Energy without the orientation distinction")
print(sandwich_energy.coefficient_table())

In [ ]:
# checks
expected_energy = (masses * velocities.scalar_norm_squared()).sum(axis=-1) / 2   # [n_frames] Scalar+
basis_rates = stack([mv.yz, mv.zx, mv.xy])                                       # [3] Rate+
assert ((energy - expected_energy[0]).abs() < 1e-12).all()
assert ((reflected_energy - energy).abs() < 1e-12).all()
assert ((rebuilt_energy - energy).abs() < 1e-12).all()
assert (((reflected_inertia - rebuilt_inertia)(basis_rates)).norm() < 1e-12).all()
assert ((expected_energy - energy).abs() < 1e-12).all()
assert ((mirrored_positions.scalar_norm_squared() - 1).abs() < 1e-12).all()
assert ((mirrored_positions[:, 1].commutator(reflected_rate) - mirrored_velocities[:, 1]).norm() < 1e-12).all()
assert ((sandwich_momentum + (mirror >> momentum)).norm() < 1e-12).all()
assert ((sandwich_energy + energy).abs() < 1e-12).all()